# Comparativa visual de horarios

Compara la asignación original del dataset con `horario_generado.csv`. Muestra conflictos y requisitos de aula, distribución semanal por día y grupo, y el horario detallado de un grupo.

**Cómo usar:** abre este notebook desde la carpeta del proyecto y selecciona **Run All**. Para cambiar el grupo de la última gráfica, modifica `GRUPO_VISUALIZAR` en su celda.

## 1. Cargar datos y validar asignaciones

Los choques se detectan comparando intervalos de tiempo que se traslapan; también se comprueban capacidad, tipo de aula y turno requerido.

In [ ]:
from pathlib import Path
from collections import Counter
from itertools import combinations

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DIAS = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes']
COLUMNAS_SESION = ['clase_id', 'grupo', 'materia', 'dia', 'hora_inicio', 'hora_fin', 'aula']

candidatos = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p for p in candidatos if (p / 'dataset' / 'clases.csv').exists()), None)
if RAIZ is None:
    raise FileNotFoundError('No se encontró dataset/clases.csv. Abre el notebook desde la carpeta del proyecto.')

clases = pd.read_csv(RAIZ / 'dataset' / 'clases.csv', encoding='utf-8-sig')
aulas = pd.read_csv(RAIZ / 'dataset' / 'aulas.csv', encoding='utf-8-sig')
original = pd.read_csv(RAIZ / 'dataset' / 'asignacion_actual.csv', encoding='utf-8-sig')
nuevo = pd.read_csv(RAIZ / 'horario_generado.csv', encoding='utf-8-sig')

def preparar(df):
    df = df.merge(
        clases[['clase_id', 'profesor', 'turno_inferido', 'alumnos_estimado',
                'tipo_aula_requerido', 'num_sesiones_requeridas']],
        on='clase_id', how='left', validate='many_to_one'
    ).merge(
        aulas[['aula_id', 'tipo', 'capacidad_estimada']].rename(
            columns={'aula_id': 'aula', 'tipo': 'tipo_aula', 'capacidad_estimada': 'capacidad_aula'}
        ),
        on='aula', how='left', validate='many_to_one'
    )
    if df[['profesor', 'tipo_aula', 'capacidad_aula']].isna().any().any():
        raise ValueError('Hay clases o aulas de los horarios que no existen en los archivos del dataset.')
    for col in ['hora_inicio', 'hora_fin']:
        df[col + '_min'] = pd.to_datetime(df[col], format='%H:%M').dt.hour * 60 + pd.to_datetime(df[col], format='%H:%M').dt.minute
    return df

horarios = {'Original': preparar(original), 'Nuevo': preparar(nuevo)}
print(f'Clases: {len(clases)} | Sesiones requeridas: {clases.num_sesiones_requeridas.sum()}')
print(f"Sesiones cargadas — original: {len(horarios['Original'])} | nuevo: {len(horarios['Nuevo'])}")

## 2. Resumen cuantitativo

Cada choque cuenta pares de sesiones que se traslapan y comparten grupo, profesor o aula. El tiempo libre se calcula entre clases consecutivas en cada día de cada grupo.

In [ ]:
def medir(df):
    choques = Counter({'Choques de grupo': 0, 'Choques de profesor': 0, 'Choques de aula': 0})
    filas = df.to_dict('records')
    for a, b in combinations(filas, 2):
        if a['dia'] != b['dia']:
            continue
        traslape = a['hora_inicio_min'] < b['hora_fin_min'] and b['hora_inicio_min'] < a['hora_fin_min']
        if not traslape:
            continue
        if a['grupo'] == b['grupo']:
            choques['Choques de grupo'] += 1
        if a['profesor'] == b['profesor']:
            choques['Choques de profesor'] += 1
        if a['aula'] == b['aula']:
            choques['Choques de aula'] += 1

    fuera_capacidad = (df.alumnos_estimado > df.capacidad_aula).sum()
    tipo_incorrecto = (df.tipo_aula != df.tipo_aula_requerido).sum()
    inicio_noche = df.hora_inicio_min >= 16 * 60
    turno_incorrecto = ((df.turno_inferido == 'Nocturno') != inicio_noche).sum()

    huecos = []
    for _, bloque in df.groupby(['grupo', 'dia']):
        intervalos = sorted(zip(bloque.hora_inicio_min, bloque.hora_fin_min))
        fin_anterior = None
        hueco_dia = 0
        for inicio, fin in intervalos:
            if fin_anterior is not None:
                hueco_dia += max(0, inicio - fin_anterior)
            fin_anterior = max(fin, fin_anterior if fin_anterior is not None else fin)
        huecos.append(hueco_dia)

    dias_por_grupo = df.groupby('grupo').dia.nunique()
    return {
        **choques,
        'Exceso de capacidad': int(fuera_capacidad),
        'Tipo de aula incorrecto': int(tipo_incorrecto),
        'Fuera del turno': int(turno_incorrecto),
        'Días por grupo (promedio)': dias_por_grupo.mean(),
        'Huecos por grupo-día (min, promedio)': sum(huecos) / len(huecos),
        'Sesiones': len(df),
    }

metricas = pd.DataFrame({nombre: medir(df) for nombre, df in horarios.items()})
display(metricas.round(2))

def claves_sesion(df):
    return Counter(map(tuple, df[COLUMNAS_SESION].astype(str).to_numpy()))

sesiones_iguales = sum((claves_sesion(original) & claves_sesion(nuevo)).values())
print(f'Sesiones que cambiaron de horario o aula: {len(original) - sesiones_iguales} de {len(original)}')
print('Nota: menos días de asistencia es mejor para compactar; los demás indicadores de errores conviene minimizarlos.')

## 3. Errores de asignación

Menos es mejor. Un choque se contabiliza por cada par de clases que coincide en el mismo recurso.

In [ ]:
indicadores_error = ['Choques de grupo', 'Choques de profesor', 'Choques de aula',
                     'Exceso de capacidad', 'Tipo de aula incorrecto', 'Fuera del turno']
ax = metricas.loc[indicadores_error].plot.barh(figsize=(10, 5), color=['#6c757d', '#168aad'])
ax.set_title('Violaciones de restricciones: original vs. nuevo')
ax.set_xlabel('Número de sesiones o pares en conflicto')
ax.set_ylabel('')
ax.grid(axis='x', alpha=.25)
ax.legend(title='Horario')
plt.tight_layout()
plt.show()

print('Sesiones nuevas que aún no cumplen el tipo de aula requerido:')
display(horarios['Nuevo'].loc[
    horarios['Nuevo'].tipo_aula != horarios['Nuevo'].tipo_aula_requerido,
    ['grupo', 'materia', 'aula', 'tipo_aula_requerido']
].reset_index(drop=True))

## 4. Distribución de clases en la semana

El primer gráfico compara cuántas sesiones hay cada día. Los mapas de calor muestran cuántos días ocupa cada grupo y cuántas sesiones tiene por día.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
distribucion = pd.DataFrame({
    nombre: df.dia.value_counts().reindex(DIAS, fill_value=0)
    for nombre, df in horarios.items()
})
distribucion.plot.bar(ax=ax, color=['#6c757d', '#168aad'], width=.8)
ax.set_title('Sesiones programadas por día')
ax.set_ylabel('Número de sesiones')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=.25)
ax.legend(title='Horario')
plt.tight_layout()
plt.show()

grupos = sorted(set(original.grupo) | set(nuevo.grupo))
fig, axes = plt.subplots(1, 2, figsize=(15, max(5, len(grupos) * .38)), sharey=True)
for ax, (nombre, df) in zip(axes, horarios.items()):
    tabla = (df.pivot_table(index='grupo', columns='dia', values='clase_id', aggfunc='count', fill_value=0)
             .reindex(index=grupos, columns=DIAS, fill_value=0))
    imagen = ax.imshow(tabla.to_numpy(), cmap='Blues', aspect='auto', vmin=0)
    ax.set_title(nombre)
    ax.set_xticks(range(len(DIAS)), DIAS, rotation=35, ha='right')
    ax.set_yticks(range(len(grupos)), grupos)
    ax.set_xlabel('Día de la semana')
    for i in range(len(grupos)):
        for j in range(len(DIAS)):
            valor = tabla.iloc[i, j]
            if valor:
                ax.text(j, i, str(valor), ha='center', va='center',
                        color='white' if valor >= 3 else '#17324d', fontsize=8)
axes[0].set_ylabel('Grupo')
fig.suptitle('Sesiones por grupo y día', y=1.01, fontsize=14)
fig.colorbar(imagen, ax=axes, label='Sesiones', shrink=.75)
plt.tight_layout()
plt.show()

## 5. Vista detallada de un grupo

Cada barra representa una clase; el eje horizontal muestra la hora y los colores distinguen los días. Cambia el grupo en la siguiente celda para revisar otros horarios.

In [ ]:
GRUPO_VISUALIZAR = '4DSA'

fig, axes = plt.subplots(2, 1, figsize=(15, 11), sharex=True)
colores = dict(zip(DIAS, plt.cm.Set2.colors[:len(DIAS)]))
for ax, (nombre, df) in zip(axes, horarios.items()):
    bloque = df[df.grupo == GRUPO_VISUALIZAR].sort_values(
        ['dia', 'hora_inicio_min', 'materia'], key=lambda col: col.map({d: i for i, d in enumerate(DIAS)}) if col.name == 'dia' else col
    ).reset_index(drop=True)
    if bloque.empty:
        raise ValueError(f'No se encontró el grupo {GRUPO_VISUALIZAR!r}. Grupos disponibles: {grupos}')
    for i, fila in bloque.iterrows():
        inicio = fila.hora_inicio_min / 60
        duracion = (fila.hora_fin_min - fila.hora_inicio_min) / 60
        ax.barh(i, duracion, left=inicio, height=.72, color=colores[fila.dia], edgecolor='white')
        etiqueta = fila.materia[:22]
        if duracion >= 1.5:
            ax.text(inicio + duracion / 2, i, etiqueta, ha='center', va='center', fontsize=7, clip_on=True)
    etiquetas = [f'{r.dia[:3]} {r.hora_inicio}-{r.hora_fin}' for r in bloque.itertuples()]
    ax.set_yticks(range(len(bloque)), etiquetas, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(nombre)
    ax.grid(axis='x', alpha=.25)
    ax.set_ylabel('Día y hora')
axes[-1].set_xlim(7, 22)
axes[-1].set_xticks(range(7, 23))
axes[-1].set_xlabel('Hora del día')
fig.suptitle(f'Distribución de clases del grupo {GRUPO_VISUALIZAR}', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Lectura rápida

La recomendación debe considerar tanto la factibilidad (sin choques y cumpliendo aulas/capacidad/turno) como la comodidad (concentrar las clases en menos días). El nuevo horario aún debe corregir cualquier requisito de aula que aparezca en la tabla de excepciones.